In [35]:
import pandas as pd
import numpy as np

In [36]:
df = pd.read_csv("ludhiana_training_data_150.csv", keep_default_na=False, na_values=[])

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   business_id             150 non-null    int64  
 1   business_name           150 non-null    object 
 2   category                150 non-null    object 
 3   location                150 non-null    object 
 4   phone_available         150 non-null    object 
 5   whatsapp_available      150 non-null    object 
 6   instagram_active        150 non-null    object 
 7   instagram_followers     150 non-null    int64  
 8   facebook_active         150 non-null    object 
 9   website_status          150 non-null    object 
 10  google_rating           150 non-null    float64
 11  google_reviews_count    150 non-null    int64  
 12  menu_catalog_available  150 non-null    object 
 13  branding_quality        150 non-null    object 
 14  years_in_business       150 non-null    in

,business_id,business_name,category,location,phone_available,whatsapp_available,instagram_active,instagram_followers,facebook_active,website_status,google_rating,google_reviews_count,menu_catalog_available,branding_quality,years_in_business,online_orders_accepted
0,1,Sharma Stationery Mart,Stationery Shop,"Khanna, Ludhiana",Yes,Yes,Yes,656,Yes,None,3.9,99,No,Low,8,No
1,2,Smart Pet Shop,Pet Store,"Machhiwara, Ludhiana",Yes,No,Yes,81,No,None,3.3,53,No,Low,11,No
2,3,Elite Photography Studio,Photography,"Doraha, Ludhiana",Yes,Yes,No,193,Yes,Basic,4.7,285,Yes,High,1,Yes
3,4,Golden Properties,Real Estate Agency,Ludhiana City,Yes,Yes,Yes,8833,Yes,Basic,4.4,236,Yes,High,2,Yes
4,5,Smart Restaurant,Restaurant,"Samrala, Ludhiana",Yes,No,No,0,No,None,4.0,54,Yes,Medium,18,No


In [37]:
print(df.isnull().sum())
print(df["website_status"].unique())
print(df["branding_quality"].unique())

business_id               0
business_name             0
category                  0
location                  0
phone_available           0
whatsapp_available        0
instagram_active          0
instagram_followers       0
facebook_active           0
website_status            0
google_rating             0
google_reviews_count      0
menu_catalog_available    0
branding_quality          0
years_in_business         0
online_orders_accepted    0
dtype: int64
['None' 'Basic' 'Good']
['Low' 'High' 'Medium']


In [38]:
yes_no_cols = ["phone_available", "whatsapp_available", "instagram_active",
               "facebook_active", "menu_catalog_available", "online_orders_accepted"]
for col in yes_no_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

In [39]:
df["website_status_score"] = df["website_status"].map({"None": 0, "Basic": 1, "Good": 2})
df["branding_quality_score"] = df["branding_quality"].map({"Low": 0, "Medium": 1, "High": 2})

In [40]:
df.isnull().sum().sum()   # must print 0

np.int64(0)

In [41]:
def google_score(row):
    s = 12 if row.google_rating >= 4.5 else 8 if row.google_rating >= 4.0 else 4
    s += 8 if row.google_reviews_count >= 150 else 5 if row.google_reviews_count >= 50 else 2
    return s

def insta_score(row):
    s = 10 if row.instagram_active == 1 else 0
    s += 10 if row.instagram_followers >= 3000 else 6 if row.instagram_followers >= 1000 else 3
    return s

df["google_pts"]   = df.apply(google_score, axis=1)
df["insta_pts"]    = df.apply(insta_score, axis=1)
df["website_pts"]  = df["website_status_score"] * 7.5
df["whatsapp_pts"] = df["whatsapp_available"] * 15
df["catalog_pts"]  = df["menu_catalog_available"] * 10
df["brand_pts"]    = df["branding_quality_score"] * 5
df["orders_pts"]   = df["online_orders_accepted"] * 10

df["lead_score"] = (df.google_pts + df.insta_pts + df.website_pts +
                     df.whatsapp_pts + df.catalog_pts + df.brand_pts + df.orders_pts).round(1)

df["lead_status"] = pd.cut(df.lead_score, bins=[-1,44,74,100], labels=["Cold","Warm","Hot"])

df.isnull().sum().sum()   # must still be 0

np.int64(0)

In [42]:
feature_cols = ["whatsapp_available", "instagram_active", "instagram_followers",
                 "facebook_active", "website_status_score", "google_rating",
                 "google_reviews_count", "menu_catalog_available",
                 "branding_quality_score", "years_in_business", "online_orders_accepted"]
# Note: phone_available dropped — it had zero variance/importance last time

X = df[feature_cols]
y = df["lead_score"]

print(X.isnull().sum().sum(), y.isnull().sum())   # both must be 0

0 0


In [43]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [44]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
model.fit(X_train, y_train)

,n_estimators,200
,criterion,'squared_error'
,max_depth,6
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [45]:
from sklearn.metrics import mean_absolute_error, r2_score

preds = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, preds))
print("R²:", r2_score(y_test, preds))

importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

MAE: 5.099611819981774
R²: 0.9589534876413237
instagram_followers       0.486977
google_rating             0.257515
website_status_score      0.097575
google_reviews_count      0.063908
whatsapp_available        0.033702
years_in_business         0.015484
menu_catalog_available    0.013642
branding_quality_score    0.013424
online_orders_accepted    0.012624
instagram_active          0.003618
facebook_active           0.001531
dtype: float64


In [46]:
import joblib
joblib.dump(model, "lead_score_model.pkl")
print("Saved.")

Saved.


In [47]:
loaded_model = joblib.load("lead_score_model.pkl")
sample = X_test.iloc[[0]]
print("Predicted:", loaded_model.predict(sample))
print("Actual:", y_test.iloc[0])

Predicted: [27.66820308]
Actual: 32.0


In [48]:
# Load the small client demo file
df_demo = pd.read_csv("sample_businesses_khanna.csv", keep_default_na=False, na_values=[])

# Apply the SAME encoding steps you used for training
yes_no_cols = ["phone_available", "whatsapp_available", "instagram_active",
               "facebook_active", "menu_catalog_available", "online_orders_accepted"]
for col in yes_no_cols:
    df_demo[col] = df_demo[col].map({"Yes": 1, "No": 0})

df_demo["website_status_score"] = df_demo["website_status"].map({"None": 0, "Basic": 1, "Good": 2})
df_demo["branding_quality_score"] = df_demo["branding_quality"].map({"Low": 0, "Medium": 1, "High": 2})

# Load the saved model
loaded_model = joblib.load("lead_score_model.pkl")

# Predict
X_demo = df_demo[feature_cols]
df_demo["predicted_lead_score"] = loaded_model.predict(X_demo).round(1)

df_demo[["business_name", "predicted_lead_score"]]

,business_name,predicted_lead_score
0,Khanna Sweets & Bakers,34.3
1,Royal Cloth House,54.9
2,Singh Hardware Store,16.6
3,Glow Up Salon & Spa,94.5
4,Khanna Electronics Hub,59.3
5,Punjabi Tadka Restaurant,72.5
6,FitZone Gym,51.8
7,Maa Durga Kirana Store,9.7
8,Trendz Footwear,40.3
9,Khanna Dental Care,67.3


In [49]:
from sklearn.ensemble import RandomForestClassifier

y_status = df["lead_status"]  # from the rule-based labels you built earlier
X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y_status, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
clf.fit(X_train2, y_train2)

joblib.dump(clf, "lead_status_classifier.pkl")

['lead_status_classifier.pkl']

In [50]:
unzip b-socio-lead-scoring.zip
cd b-socio-lead-scoring
pip install -r requirements.txt
cd src
python3 train_model.py      # retrains + saves both .pkl files automatically
streamlit run app.py        # launches the actual demo

SyntaxError: invalid syntax (4293472516.py, line 1)